In [0]:
# ============================================================
# Bronze — Source 06: Stripe API
#
# Incremental load using watermark pattern.
# Only processes S3 files newer than last successful run.
#
# Source: s3://ecommerce-lakehouse-467091806172-raw-01/source=06_stripe/
# Target: bronze.stripe catalog in Unity Catalog
#
# Tables:
#   - bronze.stripe.charges
#
# Raw format: JSON (hourly partitioned, charges array per file)
# ============================================================

import sys
sys.path.append("/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared")
from bronze_utils import get_watermark, update_watermark
from pyspark.sql.functions import col, lit, explode, max as spark_max

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
SOURCE = "06_stripe"

In [0]:
# Get watermark
watermark = get_watermark(spark, SOURCE)
print(f"Loading files modified after: {watermark}")

path = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/hour=*/stripe_charges_*.json"

# Read raw JSON and filter by file modification time
charges_raw = spark.read.format("json") \
    .option("multiLine", "false") \
    .load(path) \
    .filter(col("_metadata.file_modification_time") > lit(watermark))

file_count = charges_raw.count()

if file_count == 0:
    print(f"[{SOURCE}] No new files — skipping")
else:
    # Explode and flatten charges array
    charges_flat = charges_raw \
        .select(explode(col("charges")).alias("charge")) \
        .select(
            col("charge.payment_intent_id"),
            col("charge.amount"),
            col("charge.amount_captured"),
            col("charge.amount_refunded"),
            col("charge.currency"),
            col("charge.status"),
            col("charge.captured"),
            col("charge.description"),
            col("charge.created"),
            col("charge.created_at"),
            col("charge.metadata.customer_id").alias("customer_id"),
            col("charge.metadata.order_id").alias("order_id"),
            col("charge.failure_code"),
            col("charge.failure_message"),
            col("charge.livemode"),
            col("charge.paid"),
            col("charge.billing_details.email").alias("billing_email"),
            col("charge.billing_details.name").alias("billing_name"),
            col("charge.billing_details.address.city").alias("billing_city"),
            col("charge.billing_details.address.country").alias("billing_country"),
            col("charge.billing_details.address.postal_code").alias("billing_postcode"),
            col("charge.dispute.id").alias("dispute_id"),
            col("charge.dispute.amount").alias("dispute_amount"),
            col("charge.dispute.reason").alias("dispute_reason"),
        )

    row_count = charges_flat.count()

    spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.stripe")

    charges_flat.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("bronze.stripe.charges")

    latest_ts = charges_raw \
        .select(spark_max("_metadata.file_modification_time")).collect()[0][0]

    update_watermark(spark, SOURCE, latest_ts, row_count)
    print(f"✅ bronze.stripe.charges: {row_count} rows loaded from {file_count} files")

In [0]:
count = spark.sql("SELECT COUNT(*) as cnt FROM bronze.stripe.charges").collect()[0]['cnt']
print(f"bronze.stripe.charges: {count} rows")

spark.sql("SELECT * FROM bronze.pipeline.watermarks WHERE source = '06_stripe'").show()